In [1]:
# =========================================================
# STEP 1 — Parameters, imports, portable project root
# Single source of truth: every tunable setting lives HERE
# and is referenced by name everywhere else.
# =========================================================
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ---- Reproducibility ------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---- Portable project root ------------------------------
# Walk up from the working dir until the signature file
# data/processed/train.csv is found. Repo can move freely.
SIGNATURE = Path("data") / "processed" / "train.csv"

def find_project_root(start: Path, signature: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / signature).exists():
            return folder
    raise FileNotFoundError(
        f"Could not find {signature} above {start}. "
        "Make sure this notebook lives inside the Signify_sign2text project folder."
    )

PROJECT_ROOT = find_project_root(Path.cwd(), SIGNATURE)

# ---- Paths ----------------------------------------------
DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_CSV     = PROCESSED_DIR / "train.csv"
VAL_CSV       = PROCESSED_DIR / "val.csv"
TEST_CSV      = PROCESSED_DIR / "test.csv"

# Landmarks now live INSIDE the project (self-contained).
LANDMARK_DIR  = PROJECT_ROOT / "data" / "landmarks"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ---- Model params (locked from model_b.py) --------------
NUM_CLASSES  = 2410
SEQ_LEN      = 30
FEATURES     = 1629
HIDDEN_SIZE  = 512
NUM_LAYERS   = 3
PROJ_SIZE    = 512
LSTM_DROPOUT = 0.3
FC_DROPOUT   = 0.4

# ---- Training params ------------------------------------
BATCH_SIZE          = 64
NUM_EPOCHS          = 100
LEARNING_RATE       = 1e-3
WEIGHT_DECAY        = 1e-4
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 15     # epochs w/o val top-1 gain -> stop
NUM_WORKERS         = 0      # Windows: workers hang -> keep 0

# ---- Device ---------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Sanity print ---------------------------------------
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"LANDMARK_DIR : {LANDMARK_DIR}")
print(f"  exists?    : {LANDMARK_DIR.exists()}")
print(f"train.csv    : {TRAIN_CSV.exists()}")
print(f"val.csv      : {VAL_CSV.exists()}")
print(f"test.csv     : {TEST_CSV.exists()}")
print(f"DEVICE       : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

PROJECT_ROOT : c:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\Signify_sign2text
LANDMARK_DIR : c:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\Signify_sign2text\data\landmarks
  exists?    : True
train.csv    : True
val.csv      : True
test.csv     : True
DEVICE       : cuda
GPU          : Quadro T1000


In [2]:
# =========================================================
# STEP 2a — Load CSVs and inspect the REAL schema
# Verify column names / separator / header BEFORE building
# the landmark lookup. No assumptions about structure yet.
# =========================================================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("=== train.csv ===")
print(f"  shape   : {train_df.shape}")
print(f"  columns : {train_df.columns.tolist()}")
print()
print("dtypes:")
print(train_df.dtypes)
print()
print("first 3 rows:")
print(train_df.head(3).to_string())
print()
print(f"val.csv  shape: {val_df.shape}")
print(f"test.csv shape: {test_df.shape}")

=== train.csv ===
  shape   : (73838, 8)
  columns : ['participant', 'filename', 'video_path', 'raw_gloss', 'gloss', 'source', 'split', 'class_idx']

dtypes:
participant    object
filename       object
video_path     object
raw_gloss      object
gloss          object
source         object
split          object
class_idx       int64
dtype: object

first 3 rows:
  participant                        filename                                                                                                                                                video_path raw_gloss    gloss       source  split  class_idx
0         P48  17042711191313709-1 DOLLAR.mp4  C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Signify\data\asl_citizen\ASL_Citizen\videos\17042711191313709-1 DOLLAR.mp4   1DOLLAR  1DOLLAR  asl_citizen  train          0
1         P21    686738356933241-1 DOLLAR.mp4    C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Signify\da

In [3]:
# =========================================================
# STEP 2b — Inspect ACTUAL landmark filenames
# Goal: see how the landmark files encode the gloss segment
# (spaces? underscores? case?) before building the matcher.
# =========================================================
# List every landmark filename once, into memory.
all_npy = [p.name for p in LANDMARK_DIR.glob("*.npy")]
print(f"Total .npy files found: {len(all_npy):,}  (expected 92,904)")

# A plain sample of names.
print("\n--- 15 sample landmark filenames ---")
for name in all_npy[:15]:
    print("  ", name)

# Direct comparison: for the first 5 train rows, pull the videoid
# (everything before the first '-') and find its landmark file(s).
print("\n--- videoid match check (CSV row -> landmark file) ---")
for i in range(5):
    fn = train_df.iloc[i]["filename"]      # e.g. 17042711191313709-1 DOLLAR.mp4
    videoid = fn.split("-")[0]             # 17042711191313709
    hits = [n for n in all_npy if n.startswith(videoid)]
    print(f"  CSV filename : {fn}")
    print(f"  videoid      : {videoid}")
    print(f"  landmark hit : {hits}")
    print()

Total .npy files found: 92,904  (expected 92,904)

--- 15 sample landmark filenames ---
   000017451997373907346-LIBRARY_50fdf238.npy
   0000197996356050556-CELERY_61a3306d.npy
   000039681044643247176-PUSH_cdd8ef37.npy
   00010630150123391857-CANDY 1_949d9826.npy
   00012571487478130194-CASTLE 2_24210fc0.npy
   0001523804663805528-PASSPORT_fd8d7b8c.npy
   00015575668519507424-GOVERNMENT_ae8eecfc.npy
   00017673589179367788-OPINION 1_06cb3d62.npy
   00019179295562565812-NOT INTERESTED_6f466193.npy
   0001980973749611259-PJS_63a4daae.npy
   0002081503807414009-INHALE_2adb8819.npy
   0002339698736220086-seedCORN 2_f33c3943.npy
   00024159033092296944-LIVE 2_6b4aa0ff.npy
   0002573120929312278-WORKSHOP_c41cb279.npy
   0002600716749214804-VOMIT_c4210f45.npy

--- videoid match check (CSV row -> landmark file) ---
  CSV filename : 17042711191313709-1 DOLLAR.mp4
  videoid      : 17042711191313709
  landmark hit : ['17042711191313709-1 DOLLAR_c089aa11.npy']

  CSV filename : 686738356933241-1 

In [4]:
# =========================================================
# STEP 2c — Build {csv filename -> landmark_path} lookup,
# report match rate per split + per source, verify files.
# =========================================================

# 1) Build the lookup from every landmark file.
#    Key = filename stem minus the trailing _{hash}.
landmark_lookup = {}
n_files = 0
collisions = 0
for p in LANDMARK_DIR.glob("*.npy"):
    n_files += 1
    stem = p.stem.rsplit("_", 1)[0]        # p.stem drops '.npy'; rsplit drops hash
    if stem in landmark_lookup:
        collisions += 1
    landmark_lookup[stem] = p

print(f"Landmark files scanned : {n_files:,}")
print(f"Unique lookup keys     : {len(landmark_lookup):,}")
print(f"Collisions (same stem) : {collisions:,}\n")

# 2) Attach landmark_path to every CSV row (None if unmatched).
def lookup_path(filename):
    return landmark_lookup.get(Path(filename).stem)   # Path.stem drops '.mp4'

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df["landmark_path"] = df["filename"].map(lookup_path)

# 3) Match-rate report: overall + by source.
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    matched = df["landmark_path"].notna()
    total, hit = len(df), int(matched.sum())
    print(f"=== {name} ===  {hit:,}/{total:,} matched  ({hit/total*100:.2f}%)")
    stats = (df.assign(m=matched)
               .groupby("source")["m"].agg(["sum", "count"]))
    for src, row in stats.iterrows():
        print(f"    {src:<14}: {int(row['sum']):,}/{int(row['count']):,}")
    print()

# 4) Inspect a few UNMATCHED rows (if any) to see why.
unmatched = train_df[train_df["landmark_path"].isna()]
print(f"Unmatched train rows: {len(unmatched):,}")
if len(unmatched):
    print(unmatched[["filename", "source"]].head(10).to_string())

# 5) Spot-check a few matched .npy files: shape + sanity.
print("\n--- .npy spot check ---")
for p in train_df["landmark_path"].dropna().head(3):
    arr = np.load(p)
    print(f"  {p.name}")
    print(f"    shape={arr.shape}  dtype={arr.dtype}  "
          f"min={arr.min():.2f}  max={arr.max():.2f}  NaNs={int(np.isnan(arr).sum())}")

Landmark files scanned : 92,904
Unique lookup keys     : 92,904
Collisions (same stem) : 0

=== train ===  73,838/73,838 matched  (100.00%)
    asl_citizen   : 66,690/66,690
    wlasl         : 7,148/7,148

=== val ===  9,533/9,533 matched  (100.00%)
    asl_citizen   : 8,165/8,165
    wlasl         : 1,368/1,368

=== test ===  9,533/9,533 matched  (100.00%)
    asl_citizen   : 8,544/8,544
    wlasl         : 989/989

Unmatched train rows: 0

--- .npy spot check ---
  17042711191313709-1 DOLLAR_c089aa11.npy
    shape=(30, 1629)  dtype=float32  min=-2.05  max=3.01  NaNs=0
  686738356933241-1 DOLLAR_cd14bbbc.npy
    shape=(30, 1629)  dtype=float32  min=-2.40  max=2.90  NaNs=0
  0990039985239719-1 DOLLAR_af58950c.npy
    shape=(30, 1629)  dtype=float32  min=-1.83  max=2.57  NaNs=0


In [5]:
# =========================================================
# STEP 3 — Dataset: one (30, 1629) landmark array + its label
# =========================================================
class LandmarkDataset(Dataset):
    """Returns (x, y): x is a (30, 1629) float tensor, y is the class index."""

    def __init__(self, df):
        keep = df[df["landmark_path"].notna()]          # safety: only matched rows
        self.paths  = keep["landmark_path"].tolist()
        self.labels = keep["class_idx"].astype(int).tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        arr = np.load(self.paths[idx])                  # (30, 1629) float32 from disk
        x = torch.from_numpy(arr).float()               # numpy -> torch tensor
        y = self.labels[idx]                            # int class index
        return x, y


# ---- sanity test ----------------------------------------
train_ds = LandmarkDataset(train_df)
val_ds   = LandmarkDataset(val_df)
test_ds  = LandmarkDataset(test_df)

print(f"dataset sizes  train/val/test : {len(train_ds):,} / {len(val_ds):,} / {len(test_ds):,}")

x, y = train_ds[0]
print(f"one sample     x: {tuple(x.shape)} {x.dtype}   y: {y}")
print(f"train classes present : {train_df['class_idx'].nunique()} of {NUM_CLASSES}")

dataset sizes  train/val/test : 73,838 / 9,533 / 9,533
one sample     x: (30, 1629) torch.float32   y: 0
train classes present : 2410 of 2410


In [6]:
# =========================================================
# STEP 4 — DataLoaders (batch + shuffle + feed)
# =========================================================
pin = (DEVICE.type == "cuda")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=pin,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, drop_last=False, pin_memory=pin,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, drop_last=False, pin_memory=pin,
)

# ---- sanity test: pull one training batch ---------------
xb, yb = next(iter(train_loader))
print(f"one batch   x: {tuple(xb.shape)}  y: {tuple(yb.shape)}")
print(f"dtypes      x: {xb.dtype}   y: {yb.dtype}")
print(f"batches/epoch  train/val/test: {len(train_loader)} / {len(val_loader)} / {len(test_loader)}")

one batch   x: (64, 30, 1629)  y: (64,)
dtypes      x: torch.float32   y: torch.int64
batches/epoch  train/val/test: 1153 / 149 / 149
